# 💡 Réduction d'une matrice en forme normale d'Hermite (FNH)

La **forme normale d’Hermite (FNH)** est une forme triangulaire particulière que l’on peut obtenir à partir d’une matrice entière A en multipliant à droite par une suite de matrices **unimodulaires** U= U1*U2...*Uk telles que :

$$
AU = H
$$

où \( H \) est une matrice en **forme normale d'Hermite**, c'est-à-dire :
- triangulaire supérieure ;
- avec des coefficients diagonaux strictement positifs ;
- tous les éléments à gauche d’un élément diagonal sont strictement plus petits.

---

## ⚙️ Étapes de l'algorithme

L'algorithme procède ligne par ligne, colonne par colonne, pour transformer progressivement la matrice \( A \) en une forme FNH.

### 1. Élimination des éléments à droite de la diagonale \( (i < j) \)

Pour chaque aij, on applique l’algorithme d’Euclide étendu pour calculer :

$$
\gcd(a_{i,i}, a_{i,j}) = a_{i,i} \cdot x + a_{i,j} \cdot y
$$

Cela permet de construire une matrice unimodulaire \( M \) telle que :

$$
M(i, j, x, -\frac{b}{g}, y, \frac{a}{g})
$$


Et on applique :

$$
A \leftarrow A \cdot M,\quad U \leftarrow U \cdot M
$$

---

### 2. Réduction des éléments à gauche de la diagonale \( (j < i) \)

On souhaite réduire aij mod aii. On effectue la division euclidienne :

$$
a_{i,j} = q \cdot a_{i,i} + r,\quad \text{où } 0 \le r < |a_{i,i}|
$$

Cela revient à soustraire \( q \) fois la colonne \( i \) à la colonne \( j \), via la matrice unimodulaire :

$$
M(j, i, 1, 0, -q, 1)
$$

---

### 3. Forcer les diagonales à être positives

Si un élément diagonal devient négatif, on multiplie simplement la colonne correspondante par \( -1 \) :

$$
A[:, i] \leftarrow -A[:, i],\quad U[:, i] \leftarrow -U[:, i]
$$

---

Cela permet aussi de **résoudre un système d'équations entières** \( Ax = b \) via :
Proposition 15 Soit A de dimension m×n et rang m à coefficients entiers. Soit b un vecteur
de dimension m à coefficients entiers. Soit H = [D|0] = AU en forme normale d’Hermite
avec U unimodulaire.
1. Le syst`eme Hy = b a une solution ssi D−1b est entier et dans ce cas les solutions sont
les vecteurs de la forme :
v =
[ D−1b
z
]
où z est un vecteur d’entiers arbitraire de dimension n − m.
2. Les solutions du système Ax = b sont exactement les vecteurs U v


L'algorithme d'Euclide étendu repose sur la relation de récurrence suivante, qui provient de la division euclidienne :
pgcd⁡(a,b)=pgcd⁡(b,a mod  b)
pour retrouver x et y

L'algorithme d'Euclide a une complexité de O(log⁡(max⁡(a,b)).

In [1]:
def euclide_etendu(a, b):
    if b == 0:
        return a, 1, 0
    pgcd, x1, y1 = euclide_etendu(b, a % b)
    x = y1
    y = x1 - (a // b) * y1
    return pgcd, x, y

# Exemple
a, b = 252, 198
pgcd, x, y = euclide_etendu(a, b)
print(f"PGCD({a}, {b}) = {pgcd}, avec x = {x}, y = {y}")


PGCD(252, 198) = 18, avec x = 4, y = -5


à partir de la matrice A(n*m) nous retourne une matrice Id de taille n

In [2]:
import numpy as np
from sympy import symbols, Matrix, simplify


def matrice_Uid(A):
    """Retourne une matrice identité de taille n x n à partir de A (m*n)"""
    return np.eye(A.shape[1], dtype=int)


M(i, j, v1, v2, v3, v4) =
    identité sauf :
    M[i, i] = v1
    M[i, j] = v2
    M[j, i] = v3
    M[j, j] = v4


In [3]:

def matrice_M(A,i,j,v1,v2,v3,v4):
    """Retourne la matrice M vue en cours"""
    M = matrice_Uid(A)
    M[i,i]=v1
    M[i,j]=v2
    M[j,i]=v3
    M[j,j]=v4
    return M


In [4]:
def elimination_dessous(A, U, i, j):
    a = A[i, i]
    b = A[i, j]
    if b == 0:
        return
    g, x, y = euclide_etendu(a, b)
    
    # Matrice de transformation M appliquée à droite
    m, n = A.shape
    M = matrice_M(A,i,j,x,-b//g,y,a//g)

    A[:] = np.dot(A, M)
    U[:] = np.dot(U, M)


In [5]:
def inverse_signe_neg(A, U, i):
    """Assure que l'élément diagonal A[i, i] est positif"""
    if A[i, i] <= 0:
        A[:, i] *= -1
        U[:, i] *= -1


In [6]:
def contrainte_inf_gauche(A, U, i, j):
    """
    Réduit l'entrée a[i, j] modulo a[i, i] en soustrayant q fois la colonne i à la colonne j,
    via une multiplication par une matrice unimodulaire M (j, i, 1, 0, -q, 1)
    """
    if A[i, i] == 0:
        return  # on ne peut pas diviser
    q = A[i, j] // A[i, i]
    if q != 0:
        M = matrice_M(A, j, i, 1, 0, -q, 1)
        A[:] = A @ M
        U[:] = U @ M



In [7]:
def hermite_fn(A):
    m, n = A.shape
    U = matrice_Uid(A)  # nom corrigé

    for i in range(m):
        # Étape 1 : éliminer à droite de la diagonale
        for j in range(i + 1, n):
            if A[i, j] != 0:
                elimination_dessous(A, U, i, j)

        # Étape 2 : rendre l’élément diagonal positif
        if A[i, i] < 0:
            inverse_signe_neg(A, U, i)

        # Étape 3 : éliminer à gauche de la diagonale
        for j in range(i):
            contrainte_inf_gauche(A, U, i, j)

    return A, U  # AU = H




## 🧮 Résultat final

À la fin du processus :

- \( H \) est la forme normale d’Hermite de \( A \)
- \( U \) est unimodulaire (entière, inversible, et det={+-1)
- On a bien \( AU = H \)

In [8]:
A = np.array([
    [2, 3, 4],
    [2, 4, 6]
], dtype=int)

A_orig = A.copy()
H, U = hermite_fn(A)
print("H (forme normale d'Hermite) :\n", H)
print("U (matrice unimodulaire) :\n", U)
print("AU =\n", A_orig @ U)
print("Vérification : A @ U == H ?\n", np.all(A_orig @ U == H))

H (forme normale d'Hermite) :
 [[1 0 0]
 [0 2 0]]
U (matrice unimodulaire) :
 [[ 3 -4 -1]
 [-3  4  2]
 [ 1 -1 -1]]
AU =
 [[1 0 0]
 [0 2 0]]
Vérification : A @ U == H ?
 True


1. Le système \( Hy = b \) admet une solution **ssi** D^{-1}*b est un vecteur entier.  
   Dans ce cas, l'ensemble des solutions est :

   $$
   y = \begin{bmatrix}
   D^{-1} b \\
   z
   \end{bmatrix},\quad \text{où } z \in \mathbb{Z}^{n - m}
   $$

2. Les solutions du système \( Ax = b \) sont alors obtenues par :

$$
x = U \cdot y
$$

In [9]:
def solver(A, b):
    from sympy import symbols, Matrix, simplify

    H, U = hermite_fn(A)
    H = Matrix(H)
    U = Matrix(U)
    b = Matrix(b).reshape(len(b), 1)
    m, n = H.shape

    D = H[:, :m]  # partie carrée triangulaire supérieure
    if any(D[i, i] == 0 for i in range(m)):
        print("❌ D non inversible.")
        return None

    # Étape 2 : calcul de D⁻¹·b (manuellement, car D est triangulaire)
    y_head = []
    b_work = b.copy()
    for i in reversed(range(m)):
        if b_work[i] % D[i, i] != 0:
            print("❌ D⁻¹·b n’est pas entier → pas de solution.")
            return None
        q = b_work[i] // D[i, i]
        y_head.insert(0, q)
        b_work -= q * D[:, i]

    # Étape 3 : construire v = [D⁻¹·b ; z symbolique]
    num_libres = n - m
    if num_libres == 0:
        v = Matrix(y_head)
    else:
        z_syms = [symbols('z')] if num_libres == 1 else list(symbols(f'z0:{num_libres}'))
        v = Matrix(list(y_head) + z_syms)

    # Étape 4 : x = U · v
    x = simplify(U @ v)

    # Affichage final
    print("📐 Solution du système Ax = b :")
    print(f"x = ({', '.join(map(str, x))})ᵀ")
    return x


In [10]:
A = np.array([
    [2, 3, 4],
    [2, 4, 6]
], dtype=int)
b = np.array([3, 4])   # le b du système Hy = b

x = solver(A, b)


📐 Solution du système Ax = b :
x = (1 - z, 2*z - 1, 1 - z)ᵀ


In [11]:
A = np.array([
    [2,  4,  6,  8, 10, 12,  0,  0,  0,  0],
    [0,  1,  2,  3,  4,  5,  0,  1,  2,  3],
    [1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
    [0,  0,  1,  0, -1,  2,  0,  0,  1,  0],
    [3,  6,  9, 12, 15, 18,  1,  0, -1,  0],
    [1, -1,  0,  1, -1,  2,  2,  2,  2,  2]
], dtype=int)

b = np.array([60, 30, 10, 1, 90, 6])

x = solver(A, b)


📐 Solution du système Ax = b :
x = (2*z0 - 2*z1 + z2 + 2*z3 + 62, -21*z0 + 27*z1 - 14*z2 - 5*z3 - 16, 23*z0 - 30*z1 + 16*z2 + 5*z3, -z1, 5*z0 - 6*z1 + 3*z2 + z3, -9*z0 + 12*z1 - 6*z2 - 2*z3, 41 - z2, 2*z2 - 2*z3 - 139, 1 - z2, z3 + 61)ᵀ
